# Model Validation

Validation and edge-case experiments for passenger load, slope, energy consumption, and no-passenger behavior.

This notebook was renamed and output-stripped for repository publication. Original project data files from IETT are not included in this public repository.


In [ ]:
!pip install simpy ace_tools

In [ ]:
import simpy
import numpy as np
import random
import pandas as pd

# Yardımcı fonksiyonlar
def format_time(minutes):
    hours = int(minutes // 60)
    mins = int(minutes % 60)
    return f"{hours:02d}:{mins:02d}"

def calculate_confidence_interval(data):
    mean = np.mean(data)
    std = np.std(data, ddof=1)
    ci = 1.96 * (std / np.sqrt(len(data)))
    return mean, std, np.var(data, ddof=1), (mean - ci, mean + ci)

# RouteSegment class
class RouteSegment:
    def __init__(self, distance_km, slope_percent):
        self.distance_km = distance_km
        self.slope_percent = slope_percent

# Bus class
class Bus:
    def __init__(self, env, name, route_segments, depot_to_start, rounds, capacity, battery_capacity, co2_per_kwh, bus_area=22):
        self.env = env
        self.name = name
        self.route_segments = route_segments
        self.depot_to_start = depot_to_start
        self.rounds = rounds
        self.capacity = capacity
        self.bus_area = bus_area

        self.passengers = 0
        self.battery_capacity = battery_capacity
        self.battery_level = battery_capacity
        self.co2_per_kwh = co2_per_kwh

        self.total_co2_emissions = 0
        self.total_energy_consumed = 0
        self.passenger_counts = []
        self.routes_completed = 0
        self.total_passenger_transported = 0
        self.end_of_service_time = None
        self.total_stop_time = 0  # Duraklardaki toplam bekleme süresi

        self.action = env.process(self.run())

    def slope_correction(self, slope):
        if slope > 0:
            return 1 + (slope / 10)
        elif slope < 0:
            return max(0.7, 1 + (slope / 25))
        else:
            return 1.0

    def consumption_rate(self, passenger_mass, slope):
        base_consumptions = [0.8, 0.9, 1.0, 1.2, 1.4]
        mass_points = [0, 1000, 2000, 4000, 5700]
        base = np.interp(passenger_mass, mass_points, base_consumptions)
        slope_effect = self.slope_correction(slope)
        return max(base * slope_effect, 0.6)

    def travel_segment(self, segment, passenger_mass):
        travel_time = (segment.distance_km / 30) * 60
        yield self.env.timeout(travel_time)

        leaving = random.randint(0, min(self.passengers, 10))
        self.passengers -= leaving
        possible_boarding = random.randint(5, 20)
        boarding = min(possible_boarding, self.capacity - self.passengers)
        self.passengers += boarding

        passenger_mass = self.passengers * 70
        self.passenger_counts.append(self.passengers)
        self.total_passenger_transported += boarding

        density = self.passengers / self.bus_area
        density_points = [0, 1, 2, 4, 6]
        boarding_times = [0.89, 1.13, 1.37, 1.67, 2.03]
        alighting_times = [0.59, 0.92, 1.11, 1.89, 5.92]

        boarding_time_per_pass = np.interp(density, density_points, boarding_times)
        alighting_time_per_pass = np.interp(density, density_points, alighting_times)

        boarding_total = boarding * boarding_time_per_pass
        alighting_total = leaving * alighting_time_per_pass

        waiting_time = max(boarding_total, alighting_total) + 8.8
        self.total_stop_time += waiting_time / 60  # dakika cinsinden toplanıyor

        yield self.env.timeout(waiting_time / 60)

        consumption = self.consumption_rate(passenger_mass, segment.slope_percent) * segment.distance_km
        self.battery_level -= consumption
        self.total_energy_consumed += consumption
        co2_emission = consumption * self.co2_per_kwh
        self.total_co2_emissions += co2_emission

    def charge(self):
        yield self.env.timeout(60)
        self.battery_level = self.battery_capacity * 0.8

    def run(self):
        yield from self.travel_segment(self.depot_to_start, passenger_mass=0)

        for _ in range(self.rounds):
            route_distance = sum(seg.distance_km for seg in self.route_segments) * 2
            avg_passengers = np.mean(self.passenger_counts) if self.passenger_counts else 0
            avg_mass = avg_passengers * 70
            avg_consumption = self.consumption_rate(avg_mass, 0)
            estimated_need = avg_consumption * route_distance + 50

            if self.battery_level < estimated_need:
                return_to_depot = RouteSegment(self.depot_to_start.distance_km, -self.depot_to_start.slope_percent)
                yield from self.travel_segment(return_to_depot, passenger_mass=0)
                yield from self.charge()
                yield from self.travel_segment(self.depot_to_start, passenger_mass=0)

            for segment in self.route_segments:
                yield from self.travel_segment(segment, passenger_mass=self.passengers * 70)

            reversed_segments = [
                RouteSegment(s.distance_km, -s.slope_percent)
                for s in self.route_segments[::-1]
            ]
            for segment in reversed_segments:
                yield from self.travel_segment(segment, passenger_mass=self.passengers * 70)

            self.passengers = 0
            self.routes_completed += 1

        depot_return = RouteSegment(self.depot_to_start.distance_km, -self.depot_to_start.slope_percent)
        yield from self.travel_segment(depot_return, passenger_mass=0)
        self.end_of_service_time = self.env.now

# Simülasyon fonksiyonu
def run_simulation(seed):
    random.seed(seed)
    np.random.seed(seed)

    env = simpy.Environment(initial_time=360)
    depot_to_start = RouteSegment(2.1, -0.5)
    route_segments = [RouteSegment(d, s) for d, s in [
        (1.265, -0.16), (0.591, 0.51), (0.208, 1.44), (0.747, 0.13), (1.017, 1.08), (0.420, 3.57),
        (0.468, 2.14), (2.101, -0.24), (1.538, -0.91), (1.027, 0.58), (0.401, 3.24), (1.571, 1.08),
        (1.744, 0.11), (0.853, -0.47), (1.496, -0.53), (0.499, 1.00), (0.754, -0.80), (0.899, 0.33),
        (0.508, -0.39), (1.046, 0.00), (0.146, 2.05), (0.459, -1.53)
    ]]

    bus1 = Bus(env, "BYD K8", route_segments, depot_to_start, rounds=6, capacity=90, battery_capacity=350, co2_per_kwh=0.233, bus_area=22)
    bus2 = Bus(env, "Karsan e-Atak", route_segments, depot_to_start, rounds=6, capacity=52, battery_capacity=220, co2_per_kwh=0.233, bus_area=17)

    env.run(until=2000)

    total_stops = len(route_segments) * 2 * bus1.rounds  # Gidiş + dönüş * rounds

    return {
        # BYD K8
        "BYD K8_Passengers": bus1.total_passenger_transported,
        "BYD K8_Energy_Consumed": bus1.total_energy_consumed,
        "BYD K8_CO2": bus1.total_co2_emissions,
        "BYD K8_EndTime": bus1.end_of_service_time,
        "BYD K8_TotalServiceTime": bus1.end_of_service_time - 360,
        "BYD K8_AvgStopTime": bus1.total_stop_time / total_stops if total_stops > 0 else 0,
        "BYD K8_EnergyPerPassenger": bus1.total_energy_consumed / bus1.total_passenger_transported if bus1.total_passenger_transported > 0 else 0,

        # Karsan
        "Karsan_Passengers": bus2.total_passenger_transported,
        "Karsan_Energy_Consumed": bus2.total_energy_consumed,
        "Karsan_CO2": bus2.total_co2_emissions,
        "Karsan_EndTime": bus2.end_of_service_time,
        "Karsan_TotalServiceTime": bus2.end_of_service_time - 360,
        "Karsan_AvgStopTime": bus2.total_stop_time / total_stops if total_stops > 0 else 0,
        "Karsan_EnergyPerPassenger": bus2.total_energy_consumed / bus2.total_passenger_transported if bus2.total_passenger_transported > 0 else 0,
    }

# Simülasyonu 100 kez çalıştır
results = [run_simulation(seed) for seed in range(1, 101)]
df_results = pd.DataFrame(results)

for col in ["BYD K8_EndTime", "Karsan_EndTime"]:
    df_results[col] = df_results[col].apply(format_time)

summary_stats = {}
for column in df_results.columns:
    if "EndTime" in column:
        continue
    mean, std, var, ci = calculate_confidence_interval(df_results[column])
    summary_stats[column] = {
        "Mean": mean,
        "Std Dev": std,
        "Variance": var,
        "95% CI Lower": ci[0],
        "95% CI Upper": ci[1]
    }

summary_df = pd.DataFrame(summary_stats).T
byd_summary = summary_df.loc[summary_df.index.str.contains("BYD K8")]
karsan_summary = summary_df.loc[summary_df.index.str.contains("Karsan")]

print("📊 BYD K8 Simulation Summary")
print(byd_summary, "\n")

print("📊 Karsan e-Atak Simulation Summary")
print(karsan_summary)


In [ ]:
import simpy
import numpy as np
import random
import pandas as pd

# Helper functions
def format_time(minutes):
    hours = int(minutes // 60)
    mins = int(minutes % 60)
    return f"{hours:02d}:{mins:02d}"

def calculate_confidence_interval(data):
    mean = np.mean(data)
    std = np.std(data, ddof=1)
    ci = 1.96 * (std / np.sqrt(len(data)))
    return mean, std, np.var(data, ddof=1), (mean - ci, mean + ci)

class RouteSegment:
    def __init__(self, distance_km, slope_percent=0.0):
        self.distance_km = distance_km
        self.slope_percent = slope_percent

class Bus:
    def __init__(self, env, name, route_segments, depot_to_start, rounds, capacity, battery_capacity, co2_per_kwh, bus_area=22):
        self.env = env
        self.name = name
        self.route_segments = route_segments
        self.depot_to_start = depot_to_start
        self.rounds = rounds
        self.capacity = capacity
        self.bus_area = bus_area

        self.passengers = 0
        self.battery_capacity = battery_capacity
        self.battery_level = battery_capacity
        self.co2_per_kwh = co2_per_kwh

        self.total_co2_emissions = 0
        self.total_energy_consumed = 0
        self.passenger_counts = []
        self.routes_completed = 0
        self.total_passenger_transported = 0
        self.end_of_service_time = None

        self.action = env.process(self.run())

    def consumption_rate(self, passenger_mass, slope):
        base_consumptions = [0.8, 0.9, 1.0, 1.2, 1.4]
        mass_points = [0, 1000, 2000, 4000, 5700]
        base = np.interp(passenger_mass, mass_points, base_consumptions)
        slope_effect = 1 + (slope / 10)
        return max(base * slope_effect, 0.6)

    def travel_segment(self, segment, passenger_mass):
        travel_time = (segment.distance_km / 30) * 60
        yield self.env.timeout(travel_time)

        # No passengers in this scenario
        passenger_mass = 0
        self.passenger_counts.append(self.passengers)

        consumption = self.consumption_rate(passenger_mass, segment.slope_percent) * segment.distance_km
        self.battery_level -= consumption
        self.total_energy_consumed += consumption
        co2_emission = consumption * self.co2_per_kwh
        self.total_co2_emissions += co2_emission

    def charge(self):
        yield self.env.timeout(60)
        self.battery_level = self.battery_capacity * 0.8

    def run(self):
        yield from self.travel_segment(self.depot_to_start, passenger_mass=0)

        for _ in range(self.rounds):
            route_distance = sum(seg.distance_km for seg in self.route_segments) * 2
            avg_mass = 0
            avg_consumption = self.consumption_rate(avg_mass, 0)
            estimated_need = avg_consumption * route_distance + 50

            if self.battery_level < estimated_need:
                return_to_depot = RouteSegment(self.depot_to_start.distance_km, -self.depot_to_start.slope_percent)
                yield from self.travel_segment(return_to_depot, passenger_mass=0)
                yield from self.charge()
                yield from self.travel_segment(self.depot_to_start, passenger_mass=0)

            for segment in self.route_segments:
                yield from self.travel_segment(segment, passenger_mass=0)

            reversed_segments = [
                RouteSegment(s.distance_km, -s.slope_percent)
                for s in self.route_segments[::-1]
            ]
            for segment in reversed_segments:
                yield from self.travel_segment(segment, passenger_mass=0)

            self.routes_completed += 1

        depot_return = RouteSegment(self.depot_to_start.distance_km, -self.depot_to_start.slope_percent)
        yield from self.travel_segment(depot_return, passenger_mass=0)
        self.end_of_service_time = self.env.now

def run_simulation(seed):
    random.seed(seed)
    np.random.seed(seed)

    env = simpy.Environment(initial_time=360)
    depot_to_start = RouteSegment(2.1, -0.5)
    route_segments = [RouteSegment(d, s) for d, s in [
        (1.265, -0.16), (0.591, 0.51), (0.208, 1.44), (0.747, 0.13), (1.017, 1.08),
        (0.420, 3.57), (0.468, 2.14), (2.101, -0.24), (1.538, -0.91), (1.027, 0.58),
        (0.401, 3.24), (1.571, 1.08), (1.744, 0.11), (0.853, -0.47), (1.496, -0.53),
        (0.499, 1.00), (0.754, -0.80), (0.899, 0.33), (0.508, -0.39), (1.046, 0.00),
        (0.146, 2.05), (0.459, -1.53)
    ]]

    bus1 = Bus(env, "BYD K8", route_segments, depot_to_start, rounds=6, capacity=90, battery_capacity=350, co2_per_kwh=0.233, bus_area=22)
    bus2 = Bus(env, "Karsan e-Atak", route_segments, depot_to_start, rounds=6, capacity=52, battery_capacity=220, co2_per_kwh=0.233, bus_area=17)

    env.run(until=2000)

    return {
        "BYD K8_Passengers": bus1.total_passenger_transported,
        "BYD K8_Energy_Consumed": bus1.total_energy_consumed,
        "BYD K8_CO2": bus1.total_co2_emissions,
        "BYD K8_EndTime": bus1.end_of_service_time,
        "BYD K8_TotalServiceTime": bus1.end_of_service_time - 360,
        "Karsan_Passengers": bus2.total_passenger_transported,
        "Karsan_Energy_Consumed": bus2.total_energy_consumed,
        "Karsan_CO2": bus2.total_co2_emissions,
        "Karsan_EndTime": bus2.end_of_service_time,
        "Karsan_TotalServiceTime": bus2.end_of_service_time - 360
    }

results = [run_simulation(seed) for seed in range(1, 101)]
df_results = pd.DataFrame(results)

for col in ["BYD K8_EndTime", "Karsan_EndTime"]:
    df_results[col] = df_results[col].apply(format_time)

summary_stats = {}
for column in df_results.columns:
    if "EndTime" in column:
        continue
    mean, std, var, ci = calculate_confidence_interval(df_results[column])
    summary_stats[column] = {
        "Mean": mean,
        "Std Dev": std,
        "Variance": var,
        "95% CI Lower": ci[0],
        "95% CI Upper": ci[1]
    }

summary_df = pd.DataFrame(summary_stats).T
byd_summary = summary_df.loc[summary_df.index.str.contains("BYD K8")]
karsan_summary = summary_df.loc[summary_df.index.str.contains("Karsan")]

print("📊 BYD K8 Simulation Summary")
print(byd_summary, "\n")

print("📊 Karsan e-Atak Simulation Summary")
print(karsan_summary)


#main deriver of consumption are weight thus passenger count
#model is verified through no passenger scenario

In [ ]:
import simpy
import numpy as np
import random
import pandas as pd

# Helper functions
def format_time(minutes):
    hours = int(minutes // 60)
    mins = int(minutes % 60)
    return f"{hours:02d}:{mins:02d}"

def calculate_confidence_interval(data):
    mean = np.mean(data)
    std = np.std(data, ddof=1)
    ci = 1.96 * (std / np.sqrt(len(data)))
    return mean, std, np.var(data, ddof=1), (mean - ci, mean + ci)

class RouteSegment:
    def __init__(self, distance_km, slope_percent=0.0):
        self.distance_km = distance_km
        self.slope_percent = slope_percent

class Bus:
    def __init__(self, env, name, route_segments, depot_to_start, rounds, capacity, battery_capacity, co2_per_kwh, bus_area=22):
        self.env = env
        self.name = name
        self.route_segments = route_segments
        self.depot_to_start = depot_to_start
        self.rounds = rounds
        self.capacity = capacity
        self.bus_area = bus_area

        self.passengers = 0
        self.battery_capacity = battery_capacity
        self.battery_level = battery_capacity
        self.co2_per_kwh = co2_per_kwh

        self.total_co2_emissions = 0
        self.total_energy_consumed = 0
        self.passenger_counts = []
        self.routes_completed = 0
        self.total_passenger_transported = 0
        self.end_of_service_time = None

        self.action = env.process(self.run())

    def consumption_rate(self, passenger_mass, slope):
        base_consumptions = [0.8, 0.9, 1.0, 1.2, 1.4]
        mass_points = [0, 1000, 2000, 4000, 5700]
        base = np.interp(passenger_mass, mass_points, base_consumptions)
        slope_effect = 1 + (0 / 10)  # slope her zaman 0
        return max(base * slope_effect, 0.6)

    def travel_segment(self, segment, passenger_mass):
        travel_time = (segment.distance_km / 30) * 60
        yield self.env.timeout(travel_time)

        leaving = random.randint(0, min(self.passengers, 10))
        self.passengers -= leaving
        possible_boarding = random.randint(5, 20)
        boarding = min(possible_boarding, self.capacity - self.passengers)
        self.passengers += boarding

        passenger_mass = self.passengers * 70
        self.passenger_counts.append(self.passengers)
        self.total_passenger_transported += boarding

        density = self.passengers / self.bus_area
        density_points = [0, 1, 2, 4, 6]
        boarding_times = [0.89, 1.13, 1.37, 1.67, 2.03]
        alighting_times = [0.59, 0.92, 1.11, 1.89, 5.92]

        boarding_time_per_pass = np.interp(density, density_points, boarding_times)
        alighting_time_per_pass = np.interp(density, density_points, alighting_times)

        boarding_total = boarding * boarding_time_per_pass
        alighting_total = leaving * alighting_time_per_pass

        waiting_time = max(boarding_total, alighting_total) + 8.8
        yield self.env.timeout(waiting_time / 60)

        consumption = self.consumption_rate(passenger_mass, 0.0) * segment.distance_km
        self.battery_level -= consumption
        self.total_energy_consumed += consumption
        co2_emission = consumption * self.co2_per_kwh
        self.total_co2_emissions += co2_emission

    def charge(self):
        yield self.env.timeout(60)
        self.battery_level = self.battery_capacity * 0.8

    def run(self):
        yield from self.travel_segment(self.depot_to_start, passenger_mass=0)

        for _ in range(self.rounds):
            route_distance = sum(seg.distance_km for seg in self.route_segments) * 2
            avg_passengers = np.mean(self.passenger_counts) if self.passenger_counts else 0
            avg_mass = avg_passengers * 70
            avg_consumption = self.consumption_rate(avg_mass, 0)
            estimated_need = avg_consumption * route_distance + 50

            if self.battery_level < estimated_need:
                return_to_depot = RouteSegment(self.depot_to_start.distance_km, -self.depot_to_start.slope_percent)
                yield from self.travel_segment(return_to_depot, passenger_mass=0)
                yield from self.charge()
                yield from self.travel_segment(self.depot_to_start, passenger_mass=0)

            for segment in self.route_segments:
                yield from self.travel_segment(segment, passenger_mass=self.passengers * 70)

            reversed_segments = [
                RouteSegment(s.distance_km, -s.slope_percent)
                for s in self.route_segments[::-1]
            ]
            for segment in reversed_segments:
                yield from self.travel_segment(segment, passenger_mass=self.passengers * 70)

            self.passengers = 0
            self.routes_completed += 1

        depot_return = RouteSegment(self.depot_to_start.distance_km, -self.depot_to_start.slope_percent)
        yield from self.travel_segment(depot_return, passenger_mass=0)
        self.end_of_service_time = self.env.now

def run_simulation(seed):
    random.seed(seed)
    np.random.seed(seed)

    env = simpy.Environment(initial_time=360)
    depot_to_start = RouteSegment(2.1, 0.0)  # depot slope 0
    route_segments = [RouteSegment(d, 0.0) for d in [
        1.265, 0.591, 0.208, 0.747, 1.017, 0.420, 0.468, 2.101, 1.538,
        1.027, 0.401, 1.571, 1.744, 0.853, 1.496, 0.499, 0.754, 0.899,
        0.508, 1.046, 0.146, 0.459
    ]]

    bus1 = Bus(env, "BYD K8", route_segments, depot_to_start, rounds=6, capacity=90, battery_capacity=350, co2_per_kwh=0.233, bus_area=22)
    bus2 = Bus(env, "Karsan e-Atak", route_segments, depot_to_start, rounds=6, capacity=52, battery_capacity=220, co2_per_kwh=0.233, bus_area=17)

    env.run(until=2000)

    return {
        "BYD K8_Passengers": bus1.total_passenger_transported,
        "BYD K8_Energy_Consumed": bus1.total_energy_consumed,
        "BYD K8_CO2": bus1.total_co2_emissions,
        "BYD K8_EndTime": bus1.end_of_service_time,
        "BYD K8_TotalServiceTime": bus1.end_of_service_time - 360,
        "Karsan_Passengers": bus2.total_passenger_transported,
        "Karsan_Energy_Consumed": bus2.total_energy_consumed,
        "Karsan_CO2": bus2.total_co2_emissions,
        "Karsan_EndTime": bus2.end_of_service_time,
        "Karsan_TotalServiceTime": bus2.end_of_service_time - 360
    }

results = [run_simulation(seed) for seed in range(1, 101)]
df_results = pd.DataFrame(results)

for col in ["BYD K8_EndTime", "Karsan_EndTime"]:
    df_results[col] = df_results[col].apply(format_time)

summary_stats = {}
for column in df_results.columns:
    if "EndTime" in column:
        continue
    mean, std, var, ci = calculate_confidence_interval(df_results[column])
    summary_stats[column] = {
        "Mean": mean,
        "Std Dev": std,
        "Variance": var,
        "95% CI Lower": ci[0],
        "95% CI Upper": ci[1]
    }

summary_df = pd.DataFrame(summary_stats).T
byd_summary = summary_df.loc[summary_df.index.str.contains("BYD K8")]
karsan_summary = summary_df.loc[summary_df.index.str.contains("Karsan")]

print("📊 BYD K8 Simulation Summary")
print(byd_summary, "\n")

print("📊 Karsan e-Atak Simulation Summary")
print(karsan_summary)
